# CSI Data Collector - Out-of-Distribution (OOD) Generalization Campaign

This notebook orchestrates external validation data acquisition for the WiFi CSI presence detection system.
The campaign evaluates model inference performance under unseen physical environments and inter-node distances (other than the baseline 2.0-meter configuration).

### Protocol Timing Specifications:
- **Stabilization Window**: 60 seconds (RF link and environmental stabilization)
- **Active Window of Interest**: 60 seconds (labeled state window for evaluation)
- **Safety Buffer**: 30 seconds (ensures trailing packet capture)
- **Total Session Duration**: 60 + 60 + 30 = 150 seconds

### Enriched Metadata Fields:
Each recorded session generates companion metadata (`*_meta.json`) recording:
- `environment_id`: Unique identifier for the unseen environment
- `tx_rx_los_distance_m`: Explicit physical inter-node distance in meters
- `nodes` coordinates: Exact spatial placements (x, y coordinates and height)
- `scenario_notes`: Clutter level, obstacles, line-of-sight conditions


## 1. Imports, Constants & Target Directories

In [ ]:
import os
import sys
import time
import json
import serial
from datetime import datetime, timezone, timedelta
from pathlib import Path
import pandas as pd
import numpy as np

# Hardware and serial configuration
PORT = "/dev/ttyUSB0"  # Serial port of the RX node (ESP32-S3)
BAUD = 921600         # Must match receiver firmware baud rate

HEADER = (
    "timestamp_host,"
    "type,id,mac,rssi,rate,sig_mode,mcs,bandwidth,smoothing,"
    "not_sounding,aggregation,stbc,fec_coding,sgi,noise_floor,"
    "ampdu_cnt,channel,secondary_channel,local_timestamp,ant,"
    "sig_len,rx_state,len,first_word,data\n"
)

# Locate repository root
_p = Path.cwd()
for PROJECT_ROOT in [_p, *_p.parents]:
    if (PROJECT_ROOT / "requirements.txt").exists():
        break

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from wifi_csi.acquisition.metadata_logger import generate_session_metadata, write_session_metadata
from wifi_csi.signal.amplitudes import load_session_arrays
from wifi_csi.parsing.metadata_parser import discover_sessions

DATA_DIR = PROJECT_ROOT / "data" / "01_raw" / "generalization"
DATA_DIR.mkdir(parents=True, exist_ok=True)

def _ts_iso() -> str:
    """Return local ISO 8601 timestamp with UTC offset."""
    return datetime.now().astimezone().isoformat(timespec="seconds")

def _ts_host() -> str:
    """Host timestamp used to prefix each CSV row."""
    return datetime.now().strftime("%Y%m%dT%H%M%S.%f")

def _build_filename(session_id: str, label: str, dt: datetime) -> str:
    return f"session_{session_id}_{label}_{dt.strftime('%Y%m%d_%H%M')}"

print(f"Project root: {PROJECT_ROOT.resolve()}")
print(f"Generalization data directory: {DATA_DIR.resolve()}")
print(f"RX Serial Port: {PORT} (baud: {BAUD})")


## 2. Experimental Setup and OOD Metadata Definition

Configure the physical experimental setup parameters for the current unseen room/environment and node placement geometry.

In [ ]:
SETUP_META = {
    # Unique environment identifier
    "environment_id": "unseen_room_1",  
    "room": {
        "description": "Unseen environment with tiled floor and mixed furniture",
        "dimensions_m": {
            "east_west": 4.80,
            "north_south": 3.90,
            "ceiling_height": 2.80
        },
        "door_material": "wood",
        "window_material": "aluminum and glass",
        "notable_objects": ["desk", "wooden chair", "filing cabinet", "bookshelf"]
    },
    "nodes": {
        "tx": {
            "role": "STA (ICMP transmitter)",
            "device": "ESP32-S3-DevKitC-1",
            "tripod_height_m": 1.20,
            "pos_x_from_west_wall_m": 1.50,
            "pos_y_from_north_wall_m": 3.50,
            "mac": "1a:00:00:00:00:00"
        },
        "rx": {
            "role": "AP (CSI receiver)",
            "device": "ESP32-S3-DevKitC-1",
            "tripod_height_m": 1.20,
            "pos_x_from_west_wall_m": 1.50,
            "pos_y_from_north_wall_m": 0.50,
            "mac": "1c:db:d4:9d:93:dc",
            "serial_port": PORT,
            "baud_rate": BAUD
        },
        "tx_rx_los_distance_m": 3.00,  # Physical distance in meters
        "tx_rx_axis": "parallel to north-south axis"
    },
    "wifi": {
        "ssid": "CSI_AP",
        "channel": 11,
        "bandwidth_mhz": 40,
        "icmp_rate_hz": 30,
        "protocol": "ESP-NOW HT40"
    },
    "protocol": {
        "stabilization_s": 60,   # Link stabilization before condition starts
        "active_window_s": 60,   # Effective measurement window of interest (60 seconds)
        "buffer_end_s": 30,      # Safety buffer before stopping recording
        "subject_position": {
            "x_from_west_m": 1.50,
            "y_from_north_m": 2.00
        },
        "subject_orientation": "facing north (towards RX node)"
    },
    "scenario_notes": "Direct Line-of-Sight (LOS) test at 3.0m distance in unseen environment.",
    "labels_description": {
        "empty": "No human presence; room unoccupied, operator outside with door closed.",
        "occupied_still": "Subject standing motionless at center mark on line of sight.",
        "occupied_moving": "Subject standing at center mark with minor movements within +/- 0.30 m.",
        "occupied_p1_still": "Subject standing motionless approx. 30 cm off line of sight.",
        "occupied_p2_still": "Subject standing motionless on line of sight near TX node.",
        "occupied_p3_still": "Subject standing motionless on line of sight near RX node."
    }
}

print("Setup metadata loaded successfully.")
print(f"Environment: {SETUP_META['environment_id']}")
print(f"Distance: {SETUP_META['nodes']['tx_rx_los_distance_m']} m")


## 3. Session Parameters

Set the session ID, label, and environmental context before initiating acquisition.

In [ ]:
SESSION_ID = "GEN_A"  # Session ID: GEN_A, GEN_B, GEN_C, etc.
LABEL = "empty"        # "empty" | "occupied_still" | "occupied_moving"

STABILIZATION_S = SETUP_META["protocol"]["stabilization_s"]  # 60
ACTIVE_WINDOW_S = SETUP_META["protocol"]["active_window_s"]  # 60
BUFFER_END_S = SETUP_META["protocol"]["buffer_end_s"]        # 30
DURATION = STABILIZATION_S + ACTIVE_WINDOW_S + BUFFER_END_S  # 150 seconds
START_DELAY = 15  # Countdown seconds

ENV_COND = {
    "temperature_C": None,
    "visible_wifi_networks": None,
    "avg_rssi_dbm": None,  # Computed automatically after collection
    "door_state": "closed",
    "window_state": "closed",
    "free_notes": "Generalization session under unseen conditions"
}

_valid_labels = {
    "empty", "occupied_still", "occupied_moving"
}
assert LABEL in _valid_labels, f"Invalid LABEL '{LABEL}'. Must be in {_valid_labels}"

_dt_session = datetime.now()
_base_name = _build_filename(SESSION_ID, LABEL, _dt_session)

CSV_FILE = str(DATA_DIR / f"{_base_name}.csv")
JSON_FILE = str(DATA_DIR / f"{_base_name}_meta.json")

print(f"Session ID: {SESSION_ID}")
print(f"Label: {LABEL}")
print(f"Duration: {DURATION}s (60s stab + 60s active + 30s buffer)")
print(f"CSV Target: {CSV_FILE}")
print(f"JSON Target: {JSON_FILE}")


## 4. Live CSI Data Acquisition

Execute this cell to stream CSI packets from the ESP32 receiver node over serial.

In [ ]:
_meta_runtime = {
    "t0_recording_start": None,
    "t3_recording_stop": None,
    "total_samples": 0,
    "status": "PENDING",
    "invalidation_reason": ""
}

print(f"Starting in {START_DELAY}s... Clear the room or assume position.")
time.sleep(START_DELAY)

_t_start = datetime.now()
_meta_runtime["t0_recording_start"] = _ts_iso()
print(f"\nRECORDING STARTED: {CSV_FILE}")
print(f"t0 (Recording Start): {_meta_runtime['t0_recording_start']}")
print(f"Planned duration: {DURATION}s\n")

samples = 0
with serial.Serial(PORT, BAUD, timeout=1) as ser, open(CSV_FILE, "w", encoding="utf-8") as f:
    ser.reset_input_buffer()
    f.write(HEADER)
    t0 = time.time()
    while time.time() - t0 < DURATION:
        line = ser.readline().decode("utf-8", errors="ignore").strip()
        if line.startswith("CSI_DATA"):
            ts = _ts_host()
            f.write(f"{ts},{line}\n")
            samples += 1
            elapsed = time.time() - t0
            rate = samples / elapsed if elapsed > 0 else 0
            print(f"\r{samples:5d} samples | {elapsed:5.1f}s / {DURATION}s | {rate:4.1f} Hz", end="", flush=True)

_meta_runtime["t3_recording_stop"] = _ts_iso()
_meta_runtime["total_samples"] = samples

print(f"\n\nRecording finished: {samples} samples captured.")
print(f"t3 (Recording Stop): {_meta_runtime['t3_recording_stop']}")

_t1_auto = _t_start + timedelta(seconds=STABILIZATION_S)
_t2_auto = _t1_auto + timedelta(seconds=ACTIVE_WINDOW_S)
print(f"t1 (Active start, auto): {_t1_auto.strftime('%H:%M:%S')}")
print(f"t2 (Active end, auto):   {_t2_auto.strftime('%H:%M:%S')}")


## 5. Ground-Truth Timing & Companion Metadata Export

Construct and export the metadata JSON with all enriched experimental parameters.

In [ ]:
df_session = pd.read_csv(CSV_FILE)
rssi_mean = round(float(df_session["rssi"].mean()), 2) if "rssi" in df_session.columns and not df_session.empty else None
print(f"Session mean RSSI: {rssi_mean} dBm")
ENV_COND["avg_rssi_dbm"] = rssi_mean

SESSION_STATUS = "VALID"  # Set to "INVALID" if an anomaly occurred
INVALIDATION_REASON = ""

_date_ref = _t_start.date().isoformat()
def _to_iso(date_str: str, time_str: str) -> str:
    dt = datetime.fromisoformat(f"{date_str}T{time_str}")
    return dt.astimezone().isoformat(timespec="seconds")

T1_str = _t1_auto.strftime("%H:%M:%S")
T2_str = _t2_auto.strftime("%H:%M:%S")

_meta_runtime["t1_condition_start"] = _to_iso(_date_ref, T1_str)
_meta_runtime["t2_condition_end"] = _to_iso(_date_ref, T2_str)
_meta_runtime["status"] = SESSION_STATUS
_meta_runtime["invalidation_reason"] = INVALIDATION_REASON

metadata = {
    "schema_version": "2.1",
    "session": {
        "id": SESSION_ID,
        "label": LABEL,
        "files": {
            "csv": CSV_FILE,
            "json": JSON_FILE
        },
        "planned_duration_s": DURATION
    },
    "timing": _meta_runtime,
    "environment": {
        "environment_id": SETUP_META["environment_id"],
        "tx_rx_distance_m": SETUP_META["nodes"]["tx_rx_los_distance_m"],
        **ENV_COND,
        "scenario_notes": SETUP_META.get("scenario_notes", "")
    },
    "setup": SETUP_META
}

with open(JSON_FILE, "w", encoding="utf-8") as jf:
    json.dump(metadata, jf, indent=2, ensure_ascii=False)

print(f"Companion metadata successfully exported: {JSON_FILE}")


## 6. Immediate Session Integrity Check

Verify the session with the standardized `wifi_csi` parsing and amplitude pipeline.

In [ ]:
session_check = load_session_arrays(CSV_FILE, JSON_FILE)
print("\n--- Session Verification Summary ---")
print(f"  Session ID: {session_check['session_id']}")
print(f"  Label: {session_check['label_name']} (encoded: {session_check['label']})")
print(f"  Valid CSI Rows: {session_check['metrics']['valid_rows']}")
print(f"  Effective Sampling Rate: {session_check['effective_rate_hz']:.2f} Hz")
print(f"  Subcarrier Amplitude Matrix: {session_check['amplitude_matrix'].shape}")
print(f"  Active Condition Window: {session_check['active_start']} -> {session_check['active_end']}")
print(f"  Status: {session_check['metadata_status']}")


## 7. Campaign Overview: Generalization Sessions Recorded

In [ ]:
gen_sessions_df = discover_sessions(DATA_DIR)
if not gen_sessions_df.empty:
    print(f"Total generalization sessions found: {len(gen_sessions_df)}")
    display(gen_sessions_df[["session_id", "label_name", "metadata_status", "file_size_mb", "csv_filename"]])
else:
    print(f"No generalization sessions recorded yet in {DATA_DIR}.")
